<a href="https://colab.research.google.com/github/2303A51908/Reinforecement-Learning---B12/blob/main/2303A51908_RL_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pettingzoo==1.25.0  # Ensure pettingzoo is installed

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
from pettingzoo.mpe import simple_spread_v3  # Cooperative multi-agent env

# Simple centralized PPO actor-critic network
class MultiAgentNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU())
        self.actor = nn.Linear(hidden_dim, act_dim)
        self.critic = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x = self.shared(x)
        return self.actor(x), self.critic(x)

def compute_returns(rewards, gamma):
    returns, R = [], 0
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.FloatTensor(returns)
    return (returns - returns.mean()) / (returns.std() + 1e-8)

def ppo_update(network, optimizer, states, actions, log_probs, returns, advantages, eps_clip=0.2, ent_coef=0.01):
    new_logits, new_values = network(states)
    new_dist = Categorical(logits=new_logits)
    new_log_probs = new_dist.log_prob(actions)

    ratio = torch.exp(new_log_probs - log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantages
    policy_loss = -(torch.min(surr1, surr2) + ent_coef * new_dist.entropy()).mean()
    value_loss = F.mse_loss(new_values.squeeze(-1), returns)
    optimizer.zero_grad()
    (policy_loss + value_loss).backward()
    optimizer.step()

# Main training loop
def train_multiagent_ppo(num_episodes=500):
    env = simple_spread_v3.parallel_env()
    obs_dim = env.observation_space('agent_0').shape[0]
    act_dim = env.action_space('agent_0').n
    agents = env.possible_agents

    network = MultiAgentNetwork(obs_dim, act_dim)
    optimizer = optim.Adam(network.parameters(), lr=1e-3)

    for ep in range(num_episodes):
        # Correctly unpack reset() output: (observations, infos)
        current_obs_dict, _ = env.reset()

        # Initialize termination and truncation status for all agents.
        terminations = {agent: False for agent in agents}
        truncations = {agent: False for agent in agents}

        # Lists to store trajectory data for PPO update
        episode_states, episode_actions, episode_log_probs, episode_values = [], [], [], []
        episode_rewards = [] # Stores rewards for each (agent, step) pair

        episode_total_reward = 0 # To track total reward for printing

        while not all(terminations[agent] or truncations[agent] for agent in agents):
            actions_dict = {}
            # For each agent, get observation, compute action, store data
            for agent in agents:
                obs = torch.FloatTensor(current_obs_dict[agent])
                logits, value = network(obs)
                dist = Categorical(logits=logits)
                action = dist.sample()

                actions_dict[agent] = action.item()

                episode_states.append(obs)
                episode_actions.append(action)
                episode_log_probs.append(dist.log_prob(action))
                episode_values.append(value)

            # Step the environment with collected actions
            # Correctly unpack step() output: (observations, rewards, terminations, truncations, infos)
            next_obs_dict, reward_dict, terminations, truncations, _ = env.step(actions_dict)

            # Update current_obs_dict for the next iteration
            current_obs_dict = next_obs_dict

            # Collect rewards for PPO update
            episode_rewards.extend(list(reward_dict.values()))

            # Accumulate total reward for the episode for printing
            episode_total_reward += sum(reward_dict.values())

        # Convert collected variables to tensors for PPO update
        states = torch.stack(episode_states)
        actions = torch.stack(episode_actions)
        log_probs = torch.stack(episode_log_probs)
        values = torch.cat(episode_values).squeeze(-1)

        # Compute returns and advantages
        returns = compute_returns(episode_rewards, gamma=0.99)
        advantages = returns - values.detach()

        # Perform PPO update
        ppo_update(network, optimizer, states, actions, log_probs, returns, advantages)

        print(f"Episode {ep + 1}: Total Reward {episode_total_reward}")

if __name__ == "__main__":
    train_multiagent_ppo()


Episode 1: Total Reward -63.82358971054497
Episode 2: Total Reward -69.77344356003344
Episode 3: Total Reward -103.89690310642095
Episode 4: Total Reward -54.00125569888825
Episode 5: Total Reward -102.09268641132263
Episode 6: Total Reward -114.15438606754876
Episode 7: Total Reward -55.182138183455166
Episode 8: Total Reward -93.76554524282409
Episode 9: Total Reward -52.32993053093688
Episode 10: Total Reward -49.600133307735156
Episode 11: Total Reward -66.76043090361156
Episode 12: Total Reward -87.95165180149147
Episode 13: Total Reward -61.64089070769861
Episode 14: Total Reward -81.15237532209083
Episode 15: Total Reward -71.41772866022933
Episode 16: Total Reward -49.64603982471385
Episode 17: Total Reward -37.63079362498863
Episode 18: Total Reward -137.38894924678618
Episode 19: Total Reward -65.46263317985435
Episode 20: Total Reward -67.65954349539656
Episode 21: Total Reward -63.39886877421365
Episode 22: Total Reward -137.5378660826167
Episode 23: Total Reward -75.631315